# Walking access to transit: revised city scores and eigenvector maps

Offline review of **12 cities**, four overlapping categories, and the new saved results in `outputs/multicity/walk_access_revision/`. The categories are White alone, Black alone, Hispanic/Latino of any race, and Asian alone. Hispanic includes residents from the race-alone categories; the four counts must not be summed as a population partition.

**Geography remains explicit:** Philadelphia, Miami, Los Angeles, Salt Lake City, San Francisco, Washington DC, Houston, Detroit, Chicago, Phoenix, and Atlanta use individual populated 2020 city-limit tract portions. New York City retains its **70 custom sections**. NYC's spatial scale is different, so its scores are not directly interchangeable with tract-scale results.

Every executable cell below only reads saved local outputs and displays or checks them. It does not download data, reroute journeys, call an API, or regenerate scores. Earlier output folders are preserved. Missing revised files are reported as missing; this notebook does not substitute older results.


## Travel model and comparison scope

The former 1 km bus and 3 km rail **initial access and final egress distance caps are removed**. A person can walk farther to an actual stop, ride actual GTFS service, then walk onward. Every accepted walking second is charged, including the configured entrance/exit buffer. A transit candidate must contain at least one actual ride; a walk to a stop followed by another walk is not a transit journey.

Conservative water/land chord checks remain for access and egress. They may reject a real bridge or an indirect street route, so removing the distance caps does not establish universal real-world access. Automatic inter-stop transfer links remain within 400 m of approximate walking distance. **Explicit GTFS links, transfer restrictions, and chains of walking links remain; 400 m is not a universal limit on total transfer walking.**

Direct walking uses great-circle distance × 1.3 at 4.8 km/h. Both scenarios share this same complete direct-walking matrix, including flagged possible water crossings. The with-transit scenario adds the saved bus/rail candidates; walking-only removes transit. Other assumptions remain $15/hour value of time, an illustrative standardized $3 per transit journey, $1.25 local cost floor, and beta = 0.1 per dollar.

The saved schedules are **October 20, 2026**, except **San Francisco, August 25, 2026**. NYC uses its cached **subway/SIR only**, without buses, ferries, PATH, commuter rail, AirTrain, or the older hypothetical transit replacements. Its new-versus-old NYC comparison changes the routing method and removes proxies as well as changing access; it is not an isolated access-cap experiment. These are expected-time model estimates, not validated pedestrian itineraries or observed social contacts.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:
    display = print

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'scripts/walk_access_revision.py').exists())
BASE = ROOT / 'outputs/multicity/walk_access_revision'
CITY_NAMES = {
    'philadelphia': 'Philadelphia', 'miami': 'Miami', 'los_angeles': 'Los Angeles',
    'salt_lake_city': 'Salt Lake City', 'san_francisco': 'San Francisco',
    'washington_dc': 'Washington, DC', 'houston': 'Houston', 'detroit': 'Detroit',
    'chicago': 'Chicago', 'phoenix': 'Phoenix', 'atlanta': 'Atlanta',
    'new_york_city': 'New York City',
}
CATEGORY_NAMES = {
    'white_alone': 'White alone', 'black_alone': 'Black alone',
    'hispanic_latino_any_race': 'Hispanic, any race', 'asian_alone': 'Asian alone',
}

def read_saved_csv(name, **kwargs):
    path = BASE / name
    if not path.exists():
        print(f'Not available yet: {path.relative_to(ROOT)}')
        return pd.DataFrame()
    return pd.read_csv(path, **kwargs)

def read_saved_json(name):
    path = BASE / name
    return json.loads(path.read_text()) if path.exists() else None

scores = read_saved_csv('scenario_scores.csv')
paired = read_saved_csv('city_scores.csv')
coverage = read_saved_csv('coverage.csv')
fryer = read_saved_csv('distance_fryer_recomputed.csv')
manifest = read_saved_json('manifest.json')
verification = read_saved_json('verification.json')
if not scores.empty:
    print(f"Loaded {scores.city_id.nunique()} cities and {len(scores)} scenario/category results.")
if verification is None:
    print('Verification report not available yet; no passing validation is assumed.')
else:
    display(verification)
print('Main report:', BASE / 'index.html')
print('Map gallery:', BASE / 'maps/index.html')


## Metric definitions

Times in the cost formula are in hours; costs are in dollars:

$$C_{ij}=\max\{c_0,\min_m[p^m_{ij}+\omega t^m_{ij}]\},\qquad F_{ij}=e^{-\beta C_{ij}},$$
$$P_{ij}=\frac{F_{ij}D_j}{\sum_k F_{ik}D_k},\qquad A_r=P\,\operatorname{diag}(p_r).$$

| Quantity | Meaning |
|---|---|
| Raw PSSI | Largest eigenvalue $\lambda_r$ of $A_r$; one value per city/category/scenario |
| Normalized PSSI | $(\lambda_r-K_r)/(1-K_r)$, where $K_r$ is the category's share of all city residents |
| Node eigenvector | Right Perron vector $v_r$, normalized so $\sum_i v_{r,i}=1$ |
| Mapped node value | $n v_{r,i}$; mean across the city's nodes is 1; not a scalar PSSI normalization |
| Distance-based Fryer SSI | Separate 1 km geographic-neighbor model, computed on populated Census blocks and a tract companion |
| Adjusted Fryer | $SSI-K_r$ |
| Rescaled Fryer | $(SSI-K_r)/(1-K_r)$ |

Negative adjusted values are retained. The requested population-share subtraction is not the paper's simulated-null correction. A missing/undefined Fryer status is not silently converted to zero. The older transit-weighted majority-tract adaptation is retained only in explicitly named legacy diagnostic files; it is not the distance-based Fryer result shown here.


In [ ]:
if not scores.empty:
    display(scores[['city', 'node_scale', 'node_count', 'city_population']]
            .drop_duplicates().reset_index(drop=True))
if not paired.empty:
    wanted = ['city', 'node_scale', 'group', 'population_share_K',
              'segregation_index_with_transit', 'segregation_index_walking_only',
              'segregation_index_normalized_with_transit',
              'segregation_index_normalized_walking_only',
              'segregation_index_normalized_walking_minus_transit']
    display(paired[wanted].round(9))


## Saved PSSI comparison charts

The two travel scenarios share direct walking inputs, demographics, and economic parameters. Removing a feasible transit option cannot lower a pair's minimum generalized cost. It does not guarantee a particular direction of PSSI change because interaction rows are normalized.


In [ ]:
for filename in ['segregation_index_normalized_comparison.png', 'segregation_index_comparison.png']:
    path = BASE / filename
    if path.exists():
        fig, ax = plt.subplots(figsize=(15, 10))
        ax.imshow(plt.imread(path))
        ax.axis('off')
        plt.tight_layout()
        plt.show()
        plt.close(fig)
    else:
        print('Chart not available yet:', path.relative_to(ROOT))


## Four race-specific eigenvector maps for every city

Maps show $n v_i$ for the **walking + transit** scenario. The same fixed logarithmic color range, **0.25–4**, is used across categories and cities. Values outside the displayed range are flagged in map metadata and legends; the exact raw $v_i$ and relative $n v_i$ values remain in each map CSV. Mean 1 is a node-scale reference, not an observed probability.

Each combined image contains four separate maps. Set `CITIES_TO_DISPLAY` to a subset if you want a shorter notebook view. Every city also has four individual PNGs and a local HTML page under `maps/<city_id>/`. NYC maps show 70 custom sections; the other maps show populated city-limit tract portions.


In [ ]:
CITIES_TO_DISPLAY = list(CITY_NAMES)  # Example: ['houston', 'phoenix', 'new_york_city']
for city_id in CITIES_TO_DISPLAY:
    path = BASE / 'maps' / city_id / 'four_category_pssi_eigenvectors.png'
    if path.exists():
        print(CITY_NAMES[city_id], '|', path.parent / 'index.html')
        fig, ax = plt.subplots(figsize=(15, 12))
        ax.imshow(plt.imread(path))
        ax.axis('off')
        plt.tight_layout()
        plt.show()
        plt.close(fig)
    else:
        print('Map not available yet:', CITY_NAMES[city_id])


## Inspect one city's exact node scores and matrix checks

Identifiers are loaded as text. In the 11 tract-scale cities, `tract_geoid` is the 11-digit Census tract GEOID; in NYC it stores the section ID such as BX01. Row order must match across nodes, matrices, and eigenvectors. This cell checks existing arrays; it does not compute new routes or fit a model.


In [ ]:
CITY = 'houston'
SCENARIO = 'with_transit'
GROUP = 'white_alone'
folder = BASE / CITY / SCENARIO
matrix_path = folder / 'analysis_matrices.npz'
node_path = BASE / CITY / 'nodes.csv'
if matrix_path.exists() and node_path.exists():
    nodes = pd.read_csv(node_path, dtype={'tract_geoid': str})
    with np.load(matrix_path, allow_pickle=False) as saved:
        arrays = {key: saved[key] for key in saved.files}
    np.testing.assert_array_equal(arrays['tract_geoid'].astype(str), nodes.tract_geoid)
    np.testing.assert_allclose(arrays['P'].sum(axis=1), 1, atol=1e-10)
    np.testing.assert_allclose(arrays['right_eigenvectors'].sum(axis=0), 1, atol=1e-10)
    checks = []
    for r, group in enumerate(arrays['group_names']):
        value = arrays['eigenvalues'][r]
        vector = arrays['right_eigenvectors'][:, r]
        residual = float(np.max(np.abs(arrays['A'][r] @ vector - value * vector)))
        assert residual < 1e-9, (group, residual)
        checks.append({'group': group, 'city_eigenvalue': value,
                       'sum_right_eigenvector': float(vector.sum()), 'max_residual': residual})
    display(pd.DataFrame(checks))
    r = arrays['group_names'].tolist().index(GROUP)
    exact_nodes = nodes[['matrix_index', 'tract_geoid', 'population']].copy()
    exact_nodes['group_proportion'] = arrays['proportions'][:, r]
    exact_nodes['right_eigenvector'] = arrays['right_eigenvectors'][:, r]
    exact_nodes['mapped_n_times_v'] = len(nodes) * exact_nodes.right_eigenvector
    display(exact_nodes.sort_values('mapped_n_times_v', ascending=False).head(20))
else:
    print('Selected city/scenario arrays are not available yet:', folder)


## Distance-based Fryer scores are separate from transit

The main Fryer calculation uses populated Census **blocks**; the **tract** calculation is a companion at a different geographic scale. Distinct nodes within 1 km are geographic neighbors; there are no self-links. Each node is assigned its dominant category using the documented tie order, not a >50% majority requirement. All-neighbor weights are equal before restriction to a category, and restricted rows are not renormalized. Geographic isolates are excluded according to the documented convention.

Component Perron eigenvalues are averaged by eligible assigned-node count. This model uses geographic distances and category labels, not transit times, fares, or PSSI's population-weighted interaction matrix. Its result therefore does not change when transit access changes. The new CSV contains freshly checked 1 km calculations on the same saved geographic graphs; it reports both $SSI-K$ and $(SSI-K)/(1-K)$. Method details, local component scores, and radius sensitivity remain in `outputs/distance_fryer/`.


In [ ]:
if not fryer.empty:
    display(fryer[['city', 'fryer_node_scale', 'fryer_node_count', 'group',
                   'population_share_K', 'fryer_ssi', 'fryer_ssi_minus_K',
                   'fryer_ssi_normalized', 'status', 'change_from_previous']].round(9))
    block_view = fryer[fryer.fryer_node_scale.eq('block')]
    values = block_view.pivot(index='city', columns='group', values='fryer_ssi')
    values = values.reindex(index=list(CITY_NAMES.values()), columns=list(CATEGORY_NAMES))
    fig, ax = plt.subplots(figsize=(9, 8))
    image = ax.imshow(np.ma.masked_invalid(values.to_numpy()), vmin=0, vmax=1,
                      cmap='Blues', aspect='auto')
    ax.set_xticks(range(4), ['White alone', 'Black alone', 'Hispanic\n(any race)', 'Asian alone'])
    ax.set_yticks(range(len(values.index)), values.index)
    ax.set_title('Separate geographic model: block Fryer SSI, 1 km')
    for (i, j), value in np.ndenumerate(values.to_numpy()):
        ax.text(j, i, 'N/A' if not np.isfinite(value) else f'{value:.3f}',
                ha='center', va='center', color='white' if value > .6 else '#17304a')
    fig.colorbar(image, ax=ax, label='Raw distance-based Fryer SSI')
    plt.tight_layout()
    plt.show()
    plt.close(fig)
else:
    print('Recomputed distance-Fryer summary is not available yet.')


## Coverage, walking legs, and changes from earlier outputs

Coverage counts distinguish available transit from transit that minimizes generalized cost. Access/egress durations include the configured entrance/exit buffers; long walks are fully charged even when they make a transit option uncompetitive. A remaining lack of modeled boarding access can result from conservative barrier checks or service availability. It does not prove that no real-world transit journey exists.

For the 11 tract-scale cities, the previous-results table holds Census inputs, schedules, and costs fixed while removing access/egress distance caps. NYC's earlier router also had hypothetical transit replacements, so it is separately marked as not directly comparable.


In [ ]:
if not coverage.empty:
    display(coverage)
previous = read_saved_csv('previous_results_comparison.csv')
if not previous.empty:
    display(previous)

for name in ['city_column_dictionary.csv', 'node_column_dictionary.csv']:
    dictionary = read_saved_csv(name)
    if not dictionary.empty:
        print(name)
        display(dictionary)


## Explicit local rebuild commands (not executed by this notebook)

The following commands are documentation only. Source acquisition and Routes API collection are not enabled. Run them separately from the project root when intentionally rebuilding saved outputs:

```bash
# Recompute local access/egress and scores from cached GTFS/Census inputs.
.venv/bin/python -m scripts.walk_access_revision

# Or recompute scores from existing revised travel matrices, without rerouting.
.venv/bin/python -m scripts.walk_access_revision --analysis-only

# Or rebuild aggregate tables/plots after all per-city scenarios exist.
.venv/bin/python -m scripts.walk_access_revision --finalize-only

# Build the four-map pages/gallery and then verify the completed revision.
.venv/bin/python scripts/plot_city_eigenvectors.py
.venv/bin/python -m scripts.verify_walk_access_revision --require-maps
```

The primary report is `outputs/multicity/walk_access_revision/index.html`; the complete map gallery is `maps/index.html`. Exact matrices and node scores are under each city/scenario directory. Run all review cells safely offline; a missing verification report is reported as missing rather than treated as a passed check.
